# 0414 · XGBoost (ensamble)

**Qué hace.** Entrena y valida solo XGBoost, en las dos tareas:

| Tarea | Qué predice | Métrica para elegir |
| --- | --- | --- |
| `clf` | Probabilidad de incumplir el SLA | AP (mayor es mejor) |
| `reg` | Horas de última milla | MAE en horas (menor es mejor) |

**XGBoost** construye árboles uno tras otro; cada árbol corrige el error de los anteriores (*boosting*). En regresión la búsqueda también elige la función de pérdida (`objetivo_reg`).

**Pasos.** 1) carga lo que dejó 0411; 2) búsqueda aleatoria de hiperparámetros con validación de origen móvil; 3) validación con la mejor configuración en los tres bloques (abril, mayo, junio); 4) vista rápida de sus métricas; 5) guarda predicciones, importancias y parámetros para 0416.

**No depende de los otros cuadernos de modelos.** Solo necesita que 0411 se haya corrido. La prueba final no se toca aquí.

**Tiempo.** La búsqueda hace `N_CONFIGS_BUSQUEDA × 3 bloques × 2 tareas` ajustes sobre una submuestra de `MAX_FILAS_BUSQUEDA` filas. Para una primera corrida rápida, bajar `N_CONFIGS_BUSQUEDA` a 3.

In [0]:
# Si faltan librerías:
# !pip install duckdb lightgbm xgboost scikit-learn joblib

In [0]:
%pip install "duckdb<2" lightgbm xgboost shap
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 0. Configuración

In [0]:
import os, json, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore", category=UserWarning)

# MODULOS/ debe estar junto al cuaderno, como en 0100 (MODULOS/conexion_sql.py)
from MODULOS.modelado_temporal import (cargar_preparacion, pliegues_origen_movil, partes_del_bloque, marco_predicciones,
                                       construir_modelo, predecir, muestrear_configs, puntaje_config, mejores, params_de,
                                       importancia_pliegue, tabla_metricas)

In [0]:
MODELO = "XGBoost"
PATH_DATABRICKS = "/Volumes/proyectogrado/source/source/"
CARPETA_SALIDAS  = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/REVISION_2026_10/V3/"
CARPETA_MODELADO = f"{CARPETA_SALIDAS}MODELADO/"

N_CONFIGS_BUSQUEDA = 20        # configuraciones aleatorias por tarea
MAX_FILAS_BUSQUEDA = 300_000   # submuestra de entrenamiento en la búsqueda (no en la validación)
EJECUTAR_BUSQUEDA  = True      # False: usa los parámetros que ya guardó este cuaderno
N_PERMUTACION      = 10_000    # filas de validación para la importancia por permutación en cada bloque

In [0]:
# Hiperparámetros de base (si no hay búsqueda) y espacio de búsqueda registrado (realimentación P4)
PARAMS_BASE = dict(n_estimators=400, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8)
ESPACIO = dict(n_estimators=[200, 400, 800], learning_rate=[0.02, 0.05, 0.1], max_depth=[4, 6, 8, 10],
               min_child_weight=[1, 5, 10], subsample=[0.6, 0.8, 1.0], colsample_bytree=[0.6, 0.8, 1.0],
               reg_lambda=[1.0, 5.0, 10.0], objetivo_reg=["reg:squarederror", "reg:absoluteerror", "reg:pseudohubererror"])

## 1. Cargar la preparación de 0411

Base de modelado, historia sin futuro de cada bloque y configuración común (periodos, semilla, variables). Los pliegues se vuelven a armar con la misma función del asesor.

In [0]:
CFG, df, HIST = cargar_preparacion(CARPETA_MODELADO)
SEMILLA, CATEGORICAS, VERSIONES, TAREAS = CFG["SEMILLA"], CFG["CATEGORICAS"], CFG["VERSIONES"], CFG["TAREAS"]
PLIEGUES = {t: list(pliegues_origen_movil(df, CFG["INICIOS_VALIDACION"], CFG["FIN_DESARROLLO"], t_etiqueta=te))
            for t, _, te in TAREAS}
print(f"{len(df):,} guías · versiones: {list(VERSIONES)} · preparado el {CFG['fecha_preparacion']}")

1,588,872 guías · versiones: ['internas', 'internas+clima'] · preparado el 2026-10-24T18:56:48.281707


## 2. Búsqueda aleatoria de hiperparámetros (solo en validación)

Se prueban `N_CONFIGS_BUSQUEDA` configuraciones de `ESPACIO`. Cada una se entrena en cada bloque con una submuestra de hasta `MAX_FILAS_BUSQUEDA` filas y se evalúa en el bloque completo. Se elige la de mejor promedio entre bloques. Si la base trae la versión «internas+clima_reajustada», se busca también para ella.

In [0]:
def buscar(version, numericas):
    """Prueba todas las configuraciones de una versión de variables; una fila por configuración."""
    filas = []
    for tarea, y_col, _ in TAREAS:
        for k, cfg in enumerate(muestrear_configs(ESPACIO, N_CONFIGS_BUSQUEDA, SEMILLA + len(MODELO))):
            if tarea == "clf":
                cfg = {a: b for a, b in cfg.items() if a != "objetivo_reg"}   # solo aplica a regresión
            vals = puntaje_config(tarea, MODELO, cfg, k, PLIEGUES[tarea], HIST, df, y_col,
                                  CATEGORICAS, numericas, MAX_FILAS_BUSQUEDA, SEMILLA)
            filas.append({"version": version, "tarea": tarea, "modelo": MODELO, "config": k,
                          "metrica": "AP" if tarea == "clf" else "MAE_h", "promedio": np.mean(vals),
                          "por_bloque": [round(v, 4) for v in vals], **{f"p_{a}": b for a, b in cfg.items()}})
            print(f"{version} {tarea} config {k}: {np.mean(vals):.4f}")
    return pd.DataFrame(filas)

In [0]:
RUTA_PARAMS = f"{CARPETA_MODELADO}parametros_{MODELO}.json"
t_ini = time.time()
if EJECUTAR_BUSQUEDA:
    versiones_busqueda = ["internas"] + [v for v in VERSIONES if v == "internas+clima_reajustada"]
    tabla_busqueda = pd.concat([buscar(v, VERSIONES[v]) for v in versiones_busqueda], ignore_index=True)
    tabla_busqueda.to_pickle(f"{CARPETA_MODELADO}busqueda_{MODELO}.pkl")       # exacta, para 0416
    tabla_busqueda.to_csv(f"{CARPETA_MODELADO}busqueda_{MODELO}.csv", index=False)
    MEJORES = mejores(tabla_busqueda, {MODELO: ESPACIO})
else:
    MEJORES = json.load(open(RUTA_PARAMS))["mejores"]
print(f"Búsqueda: {time.time() - t_ini:.0f} s")
display(pd.DataFrame(MEJORES).T)

internas clf config 0: 0.3449
internas clf config 1: 0.3124
internas clf config 2: 0.3401
internas clf config 3: 0.3481
internas clf config 4: 0.3386
internas clf config 5: 0.3256
internas clf config 6: 0.3148
internas clf config 7: 0.3128
internas clf config 8: 0.3387
internas clf config 9: 0.3403
internas clf config 10: 0.3150
internas clf config 11: 0.3445
internas clf config 12: 0.3357
internas clf config 13: 0.3435
internas clf config 14: 0.3550
internas clf config 15: 0.3235
internas clf config 16: 0.3536
internas clf config 17: 0.3148
internas clf config 18: 0.3418
internas clf config 19: 0.3300
internas reg config 0: 6.8444
internas reg config 1: 34.6545
internas reg config 2: 6.8904
internas reg config 3: 6.8644
internas reg config 4: 7.8139
internas reg config 5: 8.1499
internas reg config 6: 56.8024
internas reg config 7: 6.9865
internas reg config 8: 6.9194
internas reg config 9: 6.8717
internas reg config 10: 23.7481
internas reg config 11: 6.8817
internas reg config 12: 7

n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_lambda,objetivo_reg
400.0,0.02,4.0,10.0,0.8,1.0,10.0,null
200.0,0.05,10.0,5.0,0.6,0.8,5.0,reg:absoluteerror


## 3. Validación de origen móvil con la mejor configuración

En cada versión de variables, tarea y bloque: se entrena con todo lo anterior al bloque, se predice el bloque y se guardan las predicciones. Con las variables internas se mide además la importancia por permutación (insumo de 0420).

In [0]:
pred, imp = [], []
t_ini = time.time()
for version, numericas in VERSIONES.items():
    for tarea, y_col, _ in TAREAS:
        params = params_de(MEJORES, {MODELO: PARAMS_BASE}, version, tarea, MODELO)
        for ini, tr, va in PLIEGUES[tarea]:
            Dva, Xtr, ytr, Xva, yva = partes_del_bloque(df, HIST[ini], y_col, tr, va, CATEGORICAS + numericas)
            m = construir_modelo(tarea, MODELO, numericas, CATEGORICAS, params, SEMILLA).fit(Xtr, ytr)
            pred.append(marco_predicciones(version, tarea, ini, Dva, yva).assign(modelo=MODELO, pred=predecir(m, Xva, tarea)))
            if version == "internas":
                etiquetas = {"tarea": tarea, "modelo": MODELO, "bloque": ini.date()}
                imp.append(importancia_pliegue(m, Xva, yva, tarea, N_PERMUTACION, SEMILLA, etiquetas))
            print(f"{version} · {tarea} · {ini.date()} listo ({time.time() - t_ini:.0f} s)")
pred, imp = pd.concat(pred, ignore_index=True), pd.concat(imp, ignore_index=True)

internas · clf · 2026-04-01 listo (21 s)
internas · clf · 2026-05-01 listo (47 s)
internas · clf · 2026-06-01 listo (78 s)
internas · reg · 2026-04-01 listo (100 s)
internas · reg · 2026-05-01 listo (139 s)
internas · reg · 2026-06-01 listo (172 s)
internas+clima · clf · 2026-04-01 listo (191 s)
internas+clima · clf · 2026-05-01 listo (216 s)
internas+clima · clf · 2026-06-01 listo (247 s)
internas+clima · reg · 2026-04-01 listo (268 s)
internas+clima · reg · 2026-05-01 listo (295 s)
internas+clima · reg · 2026-06-01 listo (327 s)


## 4. Vista rápida de XGBoost

Solo para revisar este modelo. La comparación con los demás se hace en 0416.

In [0]:
display(tabla_metricas(pred, ["version", "tarea", "modelo"]).round(4))
display(tabla_metricas(pred, ["version", "tarea", "modelo", "bloque"]).round(4))

version,tarea,modelo,n,prevalencia,AP,AUC_ROC,Brier,LogLoss,MAE_h,RMSE_h,P90_error_abs_h,MAE_10pct_mas_lento_h,MAPE_pct_y>=1h,cobertura_MAPE
internas,clf,XGBoost,587957,0.0709,0.3417,0.7257,0.0563,0.2286,null,null,null,null,null,null
internas,reg,XGBoost,587951,null,null,null,null,null,6.8234,37.4244,7.071,46.0325,60.6746,0.9655
internas+clima,clf,XGBoost,587957,0.0709,0.3451,0.7314,0.056,0.2242,null,null,null,null,null,null
internas+clima,reg,XGBoost,587951,null,null,null,null,null,6.8701,37.3708,7.0861,45.8564,63.5979,0.9655


version,tarea,modelo,bloque,n,prevalencia,AP,AUC_ROC,Brier,LogLoss,MAE_h,RMSE_h,P90_error_abs_h,MAE_10pct_mas_lento_h,MAPE_pct_y>=1h,cobertura_MAPE
internas,clf,XGBoost,2026-04-01,196990,0.064,0.3504,0.7236,0.0499,0.2165,null,null,null,null,null,null
internas,clf,XGBoost,2026-05-01,205304,0.0742,0.3511,0.7323,0.06,0.2448,null,null,null,null,null,null
internas,clf,XGBoost,2026-06-01,185663,0.0746,0.3395,0.7576,0.059,0.2234,null,null,null,null,null,null
internas,reg,XGBoost,2026-04-01,196989,null,null,null,null,null,6.131,37.6267,6.4979,39.0569,61.6944,0.967
internas,reg,XGBoost,2026-05-01,205301,null,null,null,null,null,6.7498,34.6815,7.6079,44.8593,60.9138,0.9646
internas,reg,XGBoost,2026-06-01,185661,null,null,null,null,null,7.6394,40.0397,7.3083,54.7238,59.3261,0.965
internas+clima,clf,XGBoost,2026-04-01,196990,0.064,0.3532,0.7301,0.0496,0.2096,null,null,null,null,null,null
internas+clima,clf,XGBoost,2026-05-01,205304,0.0742,0.3515,0.7351,0.0593,0.2385,null,null,null,null,null,null
internas+clima,clf,XGBoost,2026-06-01,185663,0.0746,0.3404,0.7581,0.0591,0.2238,null,null,null,null,null,null
internas+clima,reg,XGBoost,2026-04-01,196989,null,null,null,null,null,6.2038,37.6084,6.488,39.1568,66.4012,0.967


In [0]:
# Variables con más peso en validación (promedio de los tres bloques)
display(imp.groupby(["tarea", "variable"]).importancia.mean().sort_values(ascending=False).groupby(level=0).head(8).round(4))

tarea  variable            
reg    hist_h_zona_franja      0.5622
       hist_h_cliente          0.4527
       margen_sla_en_t0_h      0.3323
       hist_inc_zona_franja    0.3179
       n_hist_cliente          0.2137
       categoria_zona          0.2038
clf    margen_sla_en_t0_h      0.1647
reg    dia_semana              0.1549
clf    hist_inc_zona_franja    0.1098
       categoria_zona          0.0742
reg    ID_PRODUCTO             0.0741
clf    dia_semana              0.0649
       ID_PRODUCTO             0.0137
       n_hist                  0.0127
       hist_h_zona_franja      0.0126
       hist_inc_cliente        0.0064
Name: importancia, dtype: float64

## 5. Guardar para 0416

In [0]:
pred.to_parquet(f"{CARPETA_MODELADO}pred_val_{MODELO}.parquet", index=False)
imp.to_parquet(f"{CARPETA_MODELADO}importancia_{MODELO}.parquet", index=False)
with open(RUTA_PARAMS, "w") as f:
    json.dump({"modelo": MODELO, "espacio": ESPACIO, "params_base": PARAMS_BASE, "semilla": SEMILLA,
               "n_configs": N_CONFIGS_BUSQUEDA, "max_filas": MAX_FILAS_BUSQUEDA, "mejores": MEJORES}, f, indent=2, default=str)
print("Guardado en", CARPETA_MODELADO)

Guardado en /Volumes/proyectogrado/source/source/LOG_CONSOLIDADO/REVISION_2026_10/V3/MODELADO/
